In [1]:
"""
exp13 Step 03: Placebo And Information Tests Of E1M1 On DEV (PROTOCOL.md §4, step 03)

DEV = 2015-04-17 -> 2026-04-15 (exp12's candidate span; SPY and VIX cut at 2026-04-15). Reproduction asserts first.
    P1  misaligned VIX, single rule: the VIX feature table is circularly shifted by k sessions (k = round(u x rows), u
        uniform in [0.25, 0.75], seed vix_config.NULL_SEED + run) over the loaded rows; SPY stays in place; the frozen
        E1M1 is re-run. 999 runs; p1 = (1 + runs with total return >= real) / 1000.
    P2  misaligned VIX, SELECTION-AWARE (E1M1 was the best of exp12's six): in each of the same 999 shifted runs all six
        exp12 candidates are run and the MAXIMUM total return is kept; p2 = (1 + runs with that maximum >= E1M1's real
        total) / 1000. The primary information statistic of exp13 (verdict R3: p2 <= 0.05).
    P3  random re-entry after E1M1's own exits (exp04's family, matched re-entry probability), 999 runs.
    P4  random exits with E1M1's re-entry rule (exp04's family, matched exit probability), 999 runs.
Each null: median, 5th and 95th percentiles, the real value's rank and p. The runs are parallel processes, each seeded by
its index (the results do not depend on the number of processes). RUN_MODE: "validation_only" only.
Outputs: store04_experiments/exp13_e1m1_robustness/step03_placebo_data/. Trials: 4 (stage "placebo", one per test).
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
import time
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp13_e1m1_robustness import config as exp_config
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT THE MATCHED PROBABILITIES OF THE RANDOM FAMILIES
from so.core.replay_walk_forward import get_path_random_probability_dict
# IMPORT exp12's CANDIDATES
from experiments.exp12_vix_fear_reentry.rules import get_rule_candidate_list
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp13_e1m1_robustness.robustness import get_span_data_dict, get_dev_period_pdf, check_last_date_dict, get_dev_reproduction_dict, replay_rule_dict, \
    get_frozen_rule_tuple, run_parallel_null_pdf, run_vix_placebo_run_list, run_random_family_run_list, get_null_summary_dict, check_step_trial_dict
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [3]:
# DEFINE THE RUN MODE: ONLY "validation_only" EXISTS FOR THIS EXPERIMENT (NO TEST WINDOW IS EVALUATED)
RUN_MODE_STR = "validation_only"
assert RUN_MODE_STR == "validation_only"
# CHECK THE REAL DATA ONLY OUTSIDE THE SYNTHETIC SMOKE RUN (scripts/smoke_notebook.py)
CHECK_REAL_DATA_BOOL = __name__ != "__smoke__"
# DEFINE THE NUMBER OF NULL RUNS (THE SMOKE RUN ONLY CHECKS THE CODE WITH A FEW)
RUN_COUNT = exp_config.PLACEBO_RUN_COUNT if CHECK_REAL_DATA_BOOL else 4
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Run Mode:\t{RUN_MODE_STR}\tExperiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
print(f"Runs per null:\t{RUN_COUNT}\tProcesses:\t{exp_config.JOB_COUNT}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step03_placebo_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "placebo")

Run Mode:	validation_only	Experiment:	exp13_e1m1_robustness	Protocol:	1.0	Config Hash:	c2e293d595
Runs per null:	999	Processes:	8
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step03_placebo_data/
ℹ️ No earlier 'placebo' entry of exp13_e1m1_robustness in the trial log (config hash c2e293d595).


In [4]:
# CALL FUNCTION TO READ exp12's 44 REPLAY PERIODS (DATES ONLY) AND LOAD SPY AND THE VIX UP TO THEIR END (2026-04-15)
dev_period_pdf = get_dev_period_pdf(CHECK_REAL_DATA_BOOL)
dev_cutoff_date_str = str(dev_period_pdf["period_end"].iloc[-1])
dev_data_dict = get_span_data_dict(dev_cutoff_date_str)
ohlcv_array_dict, daily_pdf, vix_feature_pdf = dev_data_dict["ohlcv_array_dict"], dev_data_dict["daily_pdf"], dev_data_dict["vix_feature_pdf"]
print(f"DEV:\t{dev_period_pdf['period_start'].iloc[0]} -> {dev_cutoff_date_str}\t({len(dev_period_pdf)} periods)\tSPY sessions loaded {len(daily_pdf):,} from {dev_data_dict['first_spy_date']}")
print(f"VIX feature rows shifted by the placebo:\t{len(vix_feature_pdf):,}\t({vix_feature_pdf['date'].iloc[0]} -> {vix_feature_pdf['date'].iloc[-1]})")
dev_last_date_dict = check_last_date_dict(dev_data_dict["last_date_dict"], dev_cutoff_date_str, CHECK_REAL_DATA_BOOL)

DEV:	2015-04-17 -> 2026-04-15	(44 periods)	SPY sessions loaded 5,353 from 2005-01-03
VIX feature rows shifted by the placebo:	5,353	(2005-01-03 -> 2026-04-15)
Last date loaded:	spy	2026-04-15	(cutoff 2026-04-15)
Last date loaded:	vix_daily	2026-04-15	(cutoff 2026-04-15)
Last date loaded:	vix3m_daily	2026-04-15	(cutoff 2026-04-15)


In [5]:
"""
Reproduction (before any exp13 number), then the real values: E1M1's total return and the six exp12 candidates' totals.
"""
# CALL FUNCTION TO REPLAY AND ASSERT THE REPRODUCTION PATHS
dev_reproduction_dict = get_dev_reproduction_dict(dev_data_dict, dev_period_pdf, CHECK_REAL_DATA_BOOL)
print(f"Reproduction:\t{dev_reproduction_dict['reproduction_dict']}")
print("Reproduction asserts:\tPASSED") if CHECK_REAL_DATA_BOOL else None
date1, date2 = dev_reproduction_dict["date1"], dev_reproduction_dict["date2"]
frozen_simulation_dict = dev_reproduction_dict["frozen_simulation_dict"]
real_total = float(frozen_simulation_dict["metric_dict"]["total_return"])
# REPLAY THE SIX REAL CANDIDATES (THE REAL MAXIMUM IS E1M1's TOTAL)
candidate_dict_list = get_rule_candidate_list()
real_candidate_dict = {c["candidate_name"]: float(replay_rule_dict(ohlcv_array_dict, get_frozen_rule_tuple(daily_pdf, vix_feature_pdf, c), date1, date2)["metric_dict"]["total_return"])
                       for c in candidate_dict_list}
print(f"Real totals of the six:\t{ {name: round(100 * value, 2) for name, value in real_candidate_dict.items()} }")
assert real_candidate_dict["E1M1"] == real_total and (max(real_candidate_dict.values()) == real_total or not CHECK_REAL_DATA_BOOL)
# MATCH THE RANDOM FAMILIES TO THE FROZEN PATH
probability_dict = get_path_random_probability_dict(frozen_simulation_dict)
print(f"Matched probabilities:\t{probability_dict}")

Reproduction:	{'dev_start': '2015-04-17', 'dev_end': '2026-04-15', 'e1m1_total_pct': 314.8, 'e1m1_exit_count': 35, 'e1_total_pct': 108.03, 'e1_exit_count': 35, 'buy_hold_pct': 235.39}
Reproduction asserts:	PASSED


Real totals of the six:	{'E1M1': 314.8, 'E1M2': 195.83, 'E1M3': 132.68, 'E2M1': 235.6, 'E2M2': 167.51, 'E2M3': 168.95}
Matched probabilities:	{'exit_probability': 0.013461847546037667, 'reentry_probability': 0.21212121212121213}


In [6]:
"""
P1 and P2: the same RUN_COUNT shifted-VIX runs, all six candidates in each.
"""
# CALL FUNCTION TO RUN THE VIX-SHIFT NULL
start_time = time.time()
common_kwargs = {"ohlcv_array_dict_in": ohlcv_array_dict, "daily_pdf_in": daily_pdf, "vix_feature_pdf_in": vix_feature_pdf, "date1_in": date1, "date2_in": date2}
vix_null_pdf = run_parallel_null_pdf(run_vix_placebo_run_list, RUN_COUNT, exp_config.JOB_COUNT, candidate_dict_list_in=candidate_dict_list, **common_kwargs)
print(f"VIX-shift null:\t{len(vix_null_pdf)} runs x 6 candidates in {time.time() - start_time:.0f} s\tshift range {vix_null_pdf['shift_session_count'].min()} -> {vix_null_pdf['shift_session_count'].max()} sessions")

VIX-shift null:	999 runs x 6 candidates in 902 s	shift range 1340 -> 4014 sessions


In [7]:
"""
P3 and P4: the random families around the frozen E1M1.
"""
# CALL FUNCTION TO RUN THE TWO RANDOM FAMILIES
start_time = time.time()
random_reentry_pdf = run_parallel_null_pdf(run_random_family_run_list, RUN_COUNT, exp_config.JOB_COUNT, family_str_in="random_reentry", probability_dict_in=probability_dict, **common_kwargs)
random_exit_pdf = run_parallel_null_pdf(run_random_family_run_list, RUN_COUNT, exp_config.JOB_COUNT, family_str_in="random_exit", probability_dict_in=probability_dict, **common_kwargs)
print(f"Random families:\t2 x {RUN_COUNT} runs in {time.time() - start_time:.0f} s")

Random families:	2 x 999 runs in 306 s


In [8]:
"""
The four tests: p = (1 + runs >= real) / (runs + 1); rank 1 = the real value is above every run.
"""
# SUMMARIZE THE FOUR NULLS
placebo_dict = {"P1_vix_shift_single": get_null_summary_dict(vix_null_pdf["total_E1M1"].to_numpy(), real_total),
                "P2_vix_shift_selection_aware": get_null_summary_dict(vix_null_pdf["max_total"].to_numpy(), real_total),
                "P3_random_reentry": get_null_summary_dict(random_reentry_pdf["total_return"].to_numpy(), real_total),
                "P4_random_exit": get_null_summary_dict(random_exit_pdf["total_return"].to_numpy(), real_total)}
placebo_pdf = pd.DataFrame([{"test": name, **summary_dict} for name, summary_dict in placebo_dict.items()])
for name, summary_dict in placebo_dict.items():
    print(f"{name}:\treal {summary_dict['real_value']:.2%}\tnull median {summary_dict['null_median']:.2%}\t[5%: {summary_dict['null_p05']:.2%}, 95%: {summary_dict['null_p95']:.2%}]"
          f"\trank {summary_dict['real_rank']} of {summary_dict['run_count'] + 1}\tp = {summary_dict['p_value']:.4f}")
# WHICH CANDIDATE GAVE THE MAXIMUM IN THE SHIFTED RUNS
argmax_series = vix_null_pdf[[f"total_{c['candidate_name']}" for c in candidate_dict_list]].idxmax(axis=1).str.replace("total_", "")
print(f"Candidate giving the maximum in the shifted runs:\t{argmax_series.value_counts().to_dict()}")
# PREVIEW DATAFRAME
placebo_pdf.round(4)

P1_vix_shift_single:	real 314.80%	null median 251.56%	[5%: 185.77%, 95%: 331.64%]	rank 88 of 1000	p = 0.0880
P2_vix_shift_selection_aware:	real 314.80%	null median 254.35%	[5%: 192.50%, 95%: 331.64%]	rank 88 of 1000	p = 0.0880
P3_random_reentry:	real 314.80%	null median 125.88%	[5%: 66.23%, 95%: 209.69%]	rank 1 of 1000	p = 0.0010
P4_random_exit:	real 314.80%	null median 212.04%	[5%: 162.53%, 95%: 280.54%]	rank 13 of 1000	p = 0.0130
Candidate giving the maximum in the shifted runs:	{'E1M1': 860, 'E2M1': 126, 'E1M2': 13}


,test,run_count,real_value,null_median,null_p05,null_p95,p_value,real_rank,runs_at_or_above_real
0,P1_vix_shift_single,999,3.148,2.5156,1.8577,3.3164,0.088,88,87
1,P2_vix_shift_selection_aware,999,3.148,2.5435,1.9250,3.3164,0.088,88,87
2,P3_random_reentry,999,3.148,1.2588,0.6623,2.0969,0.001,1,0
3,P4_random_exit,999,3.148,2.1204,1.6253,2.8054,0.013,13,12


In [9]:
# SAVE THE RESULTS
write_csv_file_to_path(vix_null_pdf, f"{output_path_str}vix_shift_null_data.csv", "W")
write_csv_file_to_path(random_reentry_pdf, f"{output_path_str}random_reentry_null_data.csv", "W")
write_csv_file_to_path(random_exit_pdf, f"{output_path_str}random_exit_null_data.csv", "W")
write_csv_file_to_path(placebo_pdf, f"{output_path_str}placebo_summary_data.csv", "W")

ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step03_placebo_data/vix_shift_null_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step03_placebo_data/vix_shift_null_data.csv' Has Been Saved!
ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step03_placebo_data/random_reentry_null_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step03_placebo_data/random_reentry_null_data.csv' Has Been Saved!
ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step03_placebo_data/random_exit_null_data.csv' Not Found!	(💾 In Local F

In [10]:
# CHECK THE TRIAL BUDGET AND THE EXPECTED COUNT BEFORE LOGGING
trial_log_pdf = read_trial_log_pdf()
check_step_trial_dict(trial_log_pdf, "step03", len(placebo_dict))
# LOG EVERY TEST AS ONE PLACEBO TRIAL CARRYING ITS RUN COUNT (WINDOW COLUMNS = DEV)
logged_dict_list = []
for name, summary_dict in placebo_dict.items():
    logged_dict_list.append(log_trial_dict(exp_config, "placebo", {"step": "step03", "test": name, "candidate_name": "E1M1", "span": "dev"},
                                           {**summary_dict, "probability_dict": probability_dict, "real_candidate_total_dict": real_candidate_dict}, False,
                                           fold_row_in={"valid_start": date1, "valid_end": date2},
                                           note_str_in=f"exp13 step03 placebo / information test ({summary_dict['run_count']} null runs in one entry)"))
print(f"Trials logged:\t{len(logged_dict_list)}\tfirst logged_ts {logged_dict_list[0]['logged_ts']}\ttest_window_touched: {sorted({row['test_window_touched'] for row in logged_dict_list})}")

Trials logged:	4	first logged_ts 2026-10-09T20:23:21	test_window_touched: [False]


In [11]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp01_minute_entry,multivariate_check,1,1,0,2026-10-06T18:27:48,2026-10-06T18:27:48
1,exp01_minute_entry,signal_check,2,2,0,2026-10-05T15:17:38,2026-10-06T12:46:46
2,exp01_minute_entry,summary,1,1,0,2026-10-05T16:16:30,2026-10-05T16:16:30
3,exp01_minute_entry,validation,540,1,0,2026-10-05T15:36:53,2026-10-05T16:16:18
4,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
5,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
6,exp02_stop_reentry,summary,1,1,0,2026-10-05T14:50:29,2026-10-05T14:50:29
7,exp02_stop_reentry,validation,792,1,0,2026-10-05T14:49:18,2026-10-05T14:49:48
8,exp03_ath_exit,exploration,15,1,0,2026-10-05T14:55:04,2026-10-05T14:55:04
9,exp03_ath_exit,summary,1,1,0,2026-10-05T14:56:49,2026-10-05T14:56:49
